In [45]:
!pip install "kagglehub[pandas-datasets]"
!pip install lightgbm

In [46]:
import kagglehub
import pandas as pd
import os

# Download dataset
path = kagglehub.dataset_download("abhijitdahatonde/swiggy-restuarant-dataset")

# See files inside dataset
print(os.listdir(path))

# Load CSV
df = pd.read_csv(os.path.join(path,"swiggy.csv"))

df.head()

['swiggy.csv']


,ID,Area,City,Restaurant,Price,Avg ratings,Total ratings,Food type,Address,Delivery time
0,211,Koramangala,Bangalore,Tandoor Hut,300.0,4.4,100,"Biryani,Chinese,North Indian,South Indian",5Th Block,59
1,221,Koramangala,Bangalore,Tunday Kababi,300.0,4.1,100,"Mughlai,Lucknowi",5Th Block,56
2,246,Jogupalya,Bangalore,Kim Lee,650.0,4.4,100,Chinese,Double Road,50
3,248,Indiranagar,Bangalore,New Punjabi Hotel,250.0,3.9,500,"North Indian,Punjabi,Tandoor,Chinese",80 Feet Road,57
4,249,Indiranagar,Bangalore,Nh8,350.0,4.0,50,"Rajasthani,Gujarati,North Indian,Snacks,Desser...",80 Feet Road,63


In [47]:
import kagglehub, pandas as pd, os, numpy as np

# ---------------------------
# 1. LOAD DATASET
# ---------------------------

path = kagglehub.dataset_download("abhijitdahatonde/swiggy-restuarant-dataset")
df = pd.read_csv(os.path.join(path,"swiggy.csv"))

print("Original Shape:",df.shape)

# ---------------------------
# 2. CLEAN DATA
# ---------------------------

df.columns = df.columns.str.strip().str.lower().str.replace(" ","_")

df = df.drop_duplicates().dropna()

print("Cleaned Shape:",df.shape)

# ---------------------------
# 3. RENAME IMPORTANT FEATURES
# ---------------------------

df.rename(columns={
"id":"restaurant_id",
"avg_ratings":"avg_rating",
"total_ratings":"total_rating",
"food_type":"cuisine",
"delivery_time":"delivery_time"
}, inplace=True)

# ---------------------------
# 4. LOCATION CONTEXT FEATURES
# ---------------------------

df["city"]=df["city"].astype(str)
df["area"]=df["area"].astype(str)

# ---------------------------
# 5. SIMULATE USER ID
# (Dataset has no user id)
# ---------------------------

np.random.seed(42)
df["user_id"]=np.random.randint(1,5000,len(df))

# ---------------------------
# 6. SIMULATE ORDER TIME
# ---------------------------

df["order_time"]=pd.to_datetime(
np.random.choice(
pd.date_range("2024-01-01","2024-12-31",freq="H"),
len(df)
))

df["hour"]=df["order_time"].dt.hour

df["meal_time"]=pd.cut(
df["hour"],
bins=[0,10,15,19,24],
labels=["breakfast","lunch","evening","dinner"]
)

# ---------------------------
# 7. USER FEATURES
# ---------------------------

user_features=df.groupby("user_id").agg(

order_count=("restaurant_id","count"),
avg_order_value=("price","mean"),
favorite_city=("city","first")

).reset_index()

# ---------------------------
# 8. RESTAURANT FEATURES
# ---------------------------

restaurant_features=df.groupby("restaurant_id").agg(

avg_price=("price","mean"),
avg_rating=("avg_rating","mean"),
restaurant_popularity=("user_id","count"),
avg_delivery_time=("delivery_time","mean")

).reset_index()

# ---------------------------
# 9. CUISINE FEATURE ENGINEERING
# ---------------------------

df["num_cuisines"]=df["cuisine"].apply(lambda x: len(str(x).split(",")))

# ---------------------------
# 10. PRICE BUCKETS
# ---------------------------

df["price_bucket"]=pd.qcut(df["price"],3,labels=["budget","mid","premium"])

# ---------------------------
# 11. FINAL TRAINING DATASET
# ---------------------------

df_final=df.merge(user_features,on="user_id",how="left")\
.merge(restaurant_features,on="restaurant_id",how="left")

# ---------------------------
# 12. ADD RECOMMENDATION LABEL
# (Simulated Add-on Acceptance)
# ---------------------------

df_final["addon_acceptance"]=np.random.choice(
[0,1],
len(df_final),
p=[0.7,0.3]
)

print("\nFINAL DATASET READY")

print(df_final.head())

print("\nColumns:",df_final.columns)

print("\nFinal Shape:",df_final.shape)

Original Shape: (8680, 10)
Cleaned Shape: (8680, 10)

FINAL DATASET READY
   restaurant_id         area       city         restaurant  price  \
0            211  Koramangala  Bangalore        Tandoor Hut  300.0   
1            221  Koramangala  Bangalore      Tunday Kababi  300.0   
2            246    Jogupalya  Bangalore            Kim Lee  650.0   
3            248  Indiranagar  Bangalore  New Punjabi Hotel  250.0   
4            249  Indiranagar  Bangalore                Nh8  350.0   

   avg_rating_x  total_rating  \
0           4.4           100   
1           4.1           100   
2           4.4           100   
3           3.9           500   
4           4.0            50   

                                             cuisine       address  \
0          Biryani,Chinese,North Indian,South Indian     5Th Block   
1                                   Mughlai,Lucknowi     5Th Block   
2                                            Chinese   Double Road   
3               North Indi

/var/folders/cw/60fzrq2n52d4m1q1ly5bp5s40000gn/T/ipykernel_15966/2692565014.py:55: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  pd.date_range("2024-01-01","2024-12-31",freq="H"),


In [48]:
# ====================================
# CSAO RESTAURANT RECOMMENDATION MODEL
# ====================================

# -------------------------------
# 1. IMPORT LIBRARIES
# -------------------------------

import pandas as pd
import numpy as np

from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score


# -------------------------------
# 2. SELECT IMPORTANT FEATURES
# -------------------------------

features = [

    "price",
    "avg_rating_x",
    "restaurant_popularity",
    "avg_delivery_time",
    "order_count",
    "avg_order_value",
    "num_cuisines"

]

X = df_final[features]

y = df_final["addon_acceptance"]


# -------------------------------
# 3. TRAIN TEST SPLIT
# -------------------------------

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size = 0.2,
    random_state = 42

)


# -------------------------------
# 4. TRAIN MODEL
# -------------------------------

model = LGBMClassifier(

    n_estimators = 200,
    learning_rate = 0.05,
    max_depth = 6

)

model.fit(

    X_train,
    y_train

)


# -------------------------------
# 5. MODEL EVALUATION
# -------------------------------

predictions = model.predict_proba(X_test)[:,1]

auc = roc_auc_score(

    y_test,
    predictions

)

print("\nModel AUC Score :",auc)


# -------------------------------
# 6. GENERATE RECOMMENDATION SCORE
# -------------------------------

df_final["recommendation_score"] = model.predict_proba(

    df_final[features]

)[:,1]


# -------------------------------
# 7. USER CONTEXT FILTERING
# -------------------------------

# Example user ordering from Bangalore at Dinner

user_city = "Bangalore"
meal_context = "dinner"

filtered_data = df_final[

    (df_final["city"] == user_city)

]


# -------------------------------
# 8. SORT BY BEST RESTAURANTS
# -------------------------------

recommendations = filtered_data.sort_values(

    "recommendation_score",
    ascending = False

)


# -------------------------------
# 9. SHOW TOP RECOMMENDATIONS
# -------------------------------

top_recommendations = recommendations[[

    "restaurant_id",
    "restaurant",
    "city",
    "price",
    "avg_rating_x",
    "delivery_time",
    "recommendation_score"

]].head(10)


print("\n TOP 10 RESTAURANT RECOMMENDATIONS :\n")

print(top_recommendations)

[LightGBM] [Info] Number of positive: 2048, number of negative: 4896
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000467 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 415
[LightGBM] [Info] Number of data points in the train set: 6944, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.294931 -> initscore=-0.871555
[LightGBM] [Info] Start training from score -0.871555
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

In [49]:
print("Final AUC Score:", auc)

Final AUC Score: 0.4879089793781404


In [50]:
from sklearn.metrics import precision_score

# Sort test set by prediction probability
test_df = X_test.copy()
test_df["actual"] = y_test.values
test_df["pred_score"] = predictions

test_df = test_df.sort_values("pred_score", ascending=False)

top_k = 10

precision_at_k = test_df.head(top_k)["actual"].sum() / top_k

print("Precision@10:", precision_at_k)

Precision@10: 0.4


In [51]:
user_id_example = 10

user_data = df_final[df_final["user_id"] == user_id_example]

user_data["recommendation_score"] = model.predict_proba(user_data[features])[:,1]

user_top_recommendations = user_data.sort_values(
    "recommendation_score",
    ascending=False
)[["restaurant","city","price","avg_rating_x","recommendation_score"]].head(5)

print(user_top_recommendations)

              restaurant     city   price  avg_rating_x  recommendation_score
8640  Shree Sainath Dosa    Surat   250.0           2.9              0.331826
846        The Pizza Pie    Delhi   500.0           2.9              0.306308
616   Le Cafe Et Gateaux  Chennai   400.0           4.1              0.293422
1916    Abcos Food Plaza  Kolkata  1200.0           4.3              0.265465


/var/folders/cw/60fzrq2n52d4m1q1ly5bp5s40000gn/T/ipykernel_15966/3313305659.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  user_data["recommendation_score"] = model.predict_proba(user_data[features])[:,1]


In [52]:
df_final["addon_acceptance"] = (
    (df_final["avg_rating_x"] > 4.0) &
    (df_final["delivery_time"] < df_final["delivery_time"].median()) &
    (df_final["price"] < df_final["price"].quantile(0.75))
).astype(int)

In [53]:
df_final["price_rating_ratio"] = df_final["price"] / (df_final["avg_rating_x"] + 1)
df_final["delivery_efficiency"] = df_final["avg_rating_x"] / (df_final["delivery_time"] + 1)
df_final["popularity_score"] = df_final["restaurant_popularity"] * df_final["avg_rating_x"]

In [54]:
features = [
    "price",
    "avg_rating_x",
    "restaurant_popularity",
    "avg_delivery_time",
    "order_count",
    "avg_order_value",
    "num_cuisines",
    "price_rating_ratio",
    "delivery_efficiency",
    "popularity_score"
]

In [55]:
model = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.03,
    max_depth=8,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [56]:
X = df_final[features]
y = df_final["addon_acceptance"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict_proba(X_test)[:,1]

from sklearn.metrics import roc_auc_score
print("Improved AUC:", roc_auc_score(y_test, predictions))

[LightGBM] [Info] Number of positive: 976, number of negative: 5968
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000394 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 928
[LightGBM] [Info] Number of data points in the train set: 6944, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.140553 -> initscore=-1.810705
[LightGBM] [Info] Start training from score -1.810705
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

In [57]:
user_data = df_final[df_final["user_id"] == user_id_example]

In [58]:
user_data = df_final[df_final["user_id"] == user_id_example].copy()

In [60]:
# =====================================================
# FINAL PROFESSIONAL RESTAURANT RECOMMENDATION SYSTEM
# =====================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier


# -----------------------------------------------------
# 1. REALISTIC ACCEPTANCE LABEL (HEURISTIC)
# -----------------------------------------------------

df_final["addon_acceptance"] = (

    (df_final["avg_rating_x"] > 4.0) &
    (df_final["delivery_time"] < df_final["delivery_time"].median()) &
    (df_final["price"] < df_final["price"].quantile(0.75))

).astype(int)


# -----------------------------------------------------
# 2. ADVANCED FEATURE ENGINEERING
# -----------------------------------------------------

df_final["price_rating_ratio"] = df_final["price"] / (df_final["avg_rating_x"] + 1)

df_final["delivery_efficiency"] = (
    df_final["avg_rating_x"] / (df_final["delivery_time"] + 1)
)

df_final["popularity_score"] = (
    df_final["restaurant_popularity"] *
    df_final["avg_rating_x"]
)

df_final["value_for_money"] = (
    df_final["avg_rating_x"] / (df_final["price"] + 1)
)


# -----------------------------------------------------
# 3. FEATURE SELECTION
# -----------------------------------------------------

features = [

    "price",
    "avg_rating_x",
    "restaurant_popularity",
    "avg_delivery_time",
    "order_count",
    "avg_order_value",
    "num_cuisines",
    "price_rating_ratio",
    "delivery_efficiency",
    "popularity_score",
    "value_for_money"

]

X = df_final[features]

y = df_final["addon_acceptance"]


# -----------------------------------------------------
# 4. TRAIN TEST SPLIT
# -----------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y

)


# -----------------------------------------------------
# 5. TRAIN LIGHTGBM MODEL (FAST + OPTIMIZED)
# -----------------------------------------------------

best_model = LGBMClassifier(

    n_estimators = 300,
    learning_rate = 0.05,
    max_depth = 8,
    num_leaves = 31,
    subsample = 0.8,
    colsample_bytree = 0.8,
    random_state = 42,
    verbose = -1

)

print("Training Model...")

best_model.fit(

    X_train,
    y_train

)

print("Training Completed")


# -----------------------------------------------------
# 6. MODEL EVALUATION (AUC)
# -----------------------------------------------------

pred_probs = best_model.predict_proba(

    X_test

)[:,1]

auc_score = roc_auc_score(

    y_test,
    pred_probs

)

print("\nFinal AUC Score:", auc_score)


# -----------------------------------------------------
# 7. PRECISION @ K
# -----------------------------------------------------

test_df = X_test.copy()

test_df["actual"] = y_test.values

test_df["pred_score"] = pred_probs

test_df = test_df.sort_values(

    "pred_score",
    ascending=False

)

K = 10

precision_at_k = test_df.head(K)["actual"].sum() / K

print("Precision@10:", precision_at_k)


# -----------------------------------------------------
# 8. NDCG @ K
# -----------------------------------------------------

def ndcg_at_k(df,k):

    df = df.head(k)

    gains = df["actual"] / np.log2(np.arange(2,k+2))

    dcg = gains.sum()

    ideal = sorted(df["actual"], reverse=True)

    ideal_dcg = np.sum(

        np.array(ideal) /
        np.log2(np.arange(2,k+2))

    )

    return dcg / ideal_dcg if ideal_dcg>0 else 0


ndcg_score = ndcg_at_k(

    test_df,
    K

)

print("NDCG@10:", ndcg_score)


# -----------------------------------------------------
# 9. USER SPECIFIC RECOMMENDATIONS
# -----------------------------------------------------

user_id_example = 10

user_data = df_final[

    df_final["user_id"] == user_id_example

].copy()

user_data["recommendation_score"] = best_model.predict_proba(

    user_data[features]

)[:,1]


user_recommendations = user_data.sort_values(

    "recommendation_score",
    ascending=False

)[[

    "restaurant",
    "city",
    "price",
    "avg_rating_x",
    "delivery_time",
    "recommendation_score"

]].head(5)


print("\nTop Recommendations for User:",user_id_example)

print(user_recommendations)

Training Model...
Training Completed

Final AUC Score: 1.0
Precision@10: 1.0
NDCG@10: 1.0

Top Recommendations for User: 10
              restaurant     city   price  avg_rating_x  delivery_time  \
1916    Abcos Food Plaza  Kolkata  1200.0           4.3             63   
616   Le Cafe Et Gateaux  Chennai   400.0           4.1             68   
8640  Shree Sainath Dosa    Surat   250.0           2.9             58   
846        The Pizza Pie    Delhi   500.0           2.9             72   

      recommendation_score  
1916          1.403591e-07  
616           1.403582e-07  
8640          9.982747e-08  
846           9.765693e-08  


In [61]:
# ============================================================
# CSAO CONTEXTUAL RECOMMENDATION SYSTEM
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from lightgbm import LGBMClassifier


print("Starting CSAO Recommendation Pipeline...")


# ------------------------------------------------------------
# 1. REALISTIC ADDON ACCEPTANCE LABEL (PROBABILISTIC)
# ------------------------------------------------------------

np.random.seed(42)

prob = (

    0.4 * (df_final["avg_rating_x"] / 5) +

    0.3 * (
        1 -
        df_final["delivery_time"] /
        df_final["delivery_time"].max()
    )

    +

    0.3 * (
        1 -
        df_final["price"] /
        df_final["price"].max()
    )

)

prob = (prob - prob.min())/(prob.max()-prob.min())

df_final["addon_acceptance"] = (

    np.random.rand(len(df_final)) < prob

).astype(int)



# ------------------------------------------------------------
# 2. FEATURE ENGINEERING
# ------------------------------------------------------------

df_final["price_rating_ratio"] = (
    df_final["price"]/(df_final["avg_rating_x"]+1)
)

df_final["delivery_efficiency"] = (
    df_final["avg_rating_x"]/(df_final["delivery_time"]+1)
)

df_final["popularity_score"] = (

    df_final["restaurant_popularity"] *
    df_final["avg_rating_x"]

)

df_final["value_for_money"] = (
    df_final["avg_rating_x"]/(df_final["price"]+1)
)



# ------------------------------------------------------------
# 3. FEATURE LIST
# ------------------------------------------------------------

features=[

"price",
"avg_rating_x",
"restaurant_popularity",
"avg_delivery_time",
"order_count",
"avg_order_value",
"num_cuisines",

"price_rating_ratio",
"delivery_efficiency",
"popularity_score",
"value_for_money"

]


X=df_final[features]

y=df_final["addon_acceptance"]



# ------------------------------------------------------------
# 4. TRAIN TEST SPLIT
# ------------------------------------------------------------

X_train,X_test,y_train,y_test = train_test_split(

    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y

)



# ------------------------------------------------------------
# 5. MODEL TRAINING
# ------------------------------------------------------------

model=LGBMClassifier(

    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1

)


print("Training Model...")

model.fit(X_train,y_train)

print("Training Completed")


# ------------------------------------------------------------
# 6. MODEL EVALUATION
# ------------------------------------------------------------

pred_prob=model.predict_proba(X_test)[:,1]

auc=roc_auc_score(

    y_test,
    pred_prob

)

print("\nAUC Score :",auc)



# ------------------------------------------------------------
# 7. PRECISION@K
# ------------------------------------------------------------

test_df=X_test.copy()

test_df["actual"]=y_test.values

test_df["score"]=pred_prob

test_df=test_df.sort_values(

    "score",
    ascending=False

)

K=10

precision_at_k = test_df.head(K)["actual"].sum()/K

print("Precision@10:",precision_at_k)



# ------------------------------------------------------------
# 8. NDCG@K
# ------------------------------------------------------------

def ndcg_at_k(df,k):

    df=df.head(k)

    gains=df["actual"]/np.log2(np.arange(2,k+2))

    dcg=gains.sum()

    ideal=sorted(df["actual"],reverse=True)

    ideal_dcg=np.sum(

        np.array(ideal)/
        np.log2(np.arange(2,k+2))

    )

    return dcg/ideal_dcg if ideal_dcg>0 else 0


ndcg=ndcg_at_k(test_df,K)

print("NDCG@10:",ndcg)



# ------------------------------------------------------------
# 9. REAL TIME USER RECOMMENDATION SIMULATION
# ------------------------------------------------------------

user_id_example=10

user_data=df_final[

df_final["user_id"]==user_id_example

].copy()


user_data["recommendation_score"]=model.predict_proba(

    user_data[features]

)[:,1]


recommendations=user_data.sort_values(

    "recommendation_score",
    ascending=False

)[[

"restaurant",
"city",
"price",
"avg_rating_x",
"delivery_time",
"recommendation_score"

]].head(8)


print("\nCSAO Recommendations For User :",user_id_example)

print(recommendations)



print("\nPipeline Completed Successfully")

Starting CSAO Recommendation Pipeline...
Training Model...
Training Completed

AUC Score : 0.6702702933588477
Precision@10: 0.9
NDCG@10: 0.9196461703481418

CSAO Recommendations For User : 10
              restaurant     city   price  avg_rating_x  delivery_time  \
616   Le Cafe Et Gateaux  Chennai   400.0           4.1             68   
846        The Pizza Pie    Delhi   500.0           2.9             72   
8640  Shree Sainath Dosa    Surat   250.0           2.9             58   
1916    Abcos Food Plaza  Kolkata  1200.0           4.3             63   

      recommendation_score  
616               0.542916  
846               0.477802  
8640              0.324921  
1916              0.288487  

Pipeline Completed Successfully
